In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .appName("twibot-results")
    .config("spark.driver.memory", "1g")
    .getOrCreate())
out = "/home/jovyan/work/parquet"
print(spark.version)

In [ ]:
import os
import matplotlib.pyplot as plt

p = spark.read.parquet(f"{out}/community_scores").toPandas()
p["label"] = p["label"].astype(str)
os.makedirs("/home/jovyan/work/results", exist_ok=True)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(p["coord_score"], p["lift"], s=(p["size"] ** 0.5) * 12, alpha=0.6)
ax.axhline(1.0, linestyle="--", linewidth=1)
ax.set_xlabel("Coordination score (label-free signals)")
ax.set_ylabel("Bot enrichment (lift vs core baseline of 6.2%)")
ax.set_title("Communities: coordination score vs bot enrichment")

# label the 8 highest-scoring communities with the last 4 digits of the ID
for _, r in p.sort_values("coord_score", ascending=False).head(8).iterrows():
    ax.annotate(r["label"][-4:], (r["coord_score"], r["lift"]),
                fontsize=8, xytext=(4, 4), textcoords="offset points")

fig.text(0.01, -0.02,
         "Bubble area scales with community size. Labels are the last 4 digits of the community ID.",
         fontsize=8)
fig.savefig("/home/jovyan/work/results/score_vs_lift.png", dpi=150, bbox_inches="tight")
plt.show()

cols = ["label", "size", "coord_score", "lift", "bot_ratio", "density",
        "burst_share", "no_desc_share", "median_ratio"]
tbl = (p.sort_values("coord_score", ascending=False)[cols]
         .reset_index(drop=True))
tbl.to_csv("/home/jovyan/work/results/flagged_communities.csv", index=False)
tbl.head(10)